## 02 — Vector Search with SimlarEngine

In this notebook we index 500 Quora questions using locally-computed sentence embeddings and search by semantic similarity.

**What we cover**
- Attaching a `sentence-transformers` embedder so `SimlarEngine` embeds documents and queries for you
- Incremental adds — growing the index after initial construction
- Single- and multi-query search via `search()` — returns ranked `SearchResult` objects
- Updating and deleting documents by ID
- Saving the index to disk and reloading it

In [1]:
%pip install -q datasets sentence-transformers simlar

Note: you may need to restart the kernel to use updated packages.


## Load the dataset and set up an embedder

We use [`BeIR/quora`](https://huggingface.co/datasets/BeIR/quora) — a corpus of 500,000 community questions.
We take the first 500 questions and embed them with `all-MiniLM-L6-v2` (384-dimensional embeddings).

`CallableEmbedder` wraps the model's `encode` function. With it attached, the index embeds document text on `add()` / `update()` and query text on `search()`, L2-normalised and in batches.

The first 200 seed the initial index; the remaining 300 are added incrementally to show live growth.

In [2]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from simlar import CallableEmbedder, SimlarEngine

ds = load_dataset("BeIR/quora", "corpus", split="corpus[:500]")
all_texts = ds["text"]
all_ids   = ds["_id"]

model    = SentenceTransformer("all-MiniLM-L6-v2")
embedder = CallableEmbedder(model.encode, normalize=True, batch_size=256)

id_to_text = dict(zip(all_ids, all_texts))

print(f"Loaded {len(all_texts)} questions")
print(f"Sample: {all_texts[0][:100]}")

/home/aramirez/anaconda3/envs/simlar/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13381.24it/s]

Loaded 500 questions
Sample: What is the step by step guide to invest in share market in india?


## Build the index

With an embedder attached, `add()` takes IDs and texts. If you already have embeddings, pass them as `vectors=` (an `(n, dim)` float32 matrix) and the embedder is skipped.

In [3]:
idx = SimlarEngine(embedder=embedder)
idx.add(ids=list(all_ids[:200]), texts=list(all_texts[:200]))

print(f"Index size : {idx.size}")
print(f"Is trained : {idx.is_trained}")

Index size : 200
Is trained : True


## Incremental add

Call `add()` again at any time to grow the index without rebuilding it.

In [4]:
idx.add(ids=list(all_ids[200:]), texts=list(all_texts[200:]))
print(f"After incremental add: {idx.size} documents")

After incremental add: 500 documents


## Single-query search and Multi-query search

`search(query_text=..., k=...)` embeds the query and returns a list of `SearchResult` objects, each with `.id`, `.rank`, and `.score`. A list of query strings is a batch and returns one result list per query. You can still pass a precomputed `(n, dim)` query vector as the first argument instead.

In [5]:
QUERY_TEXT = "What is the best programming language to learn first?"

print(f"Query: '{QUERY_TEXT}'\n")
for r in idx.search(query_text=QUERY_TEXT, k=5):
    print(f"  rank={r.rank}  score={r.score:.4f}  →  {id_to_text[r.id]}")

Query: 'What is the best programming language to learn first?'

  rank=0  score=0.6799  →  How do I learn a computer language like java?
  rank=1  score=0.6108  →  What is Java programming? How To Learn Java Programming Language ?
  rank=2  score=0.6012  →  What's the best way to start learning robotics?
  rank=3  score=0.5454  →  What is the best way to start robotics? Which is the best development board that I can start working on it?
  rank=4  score=0.5377  →  Which test series is the best for GATE computer science stream?


In [6]:
QUERIES = [
    "How do I invest in the stock market?",
    "What are the best ways to lose weight?",
    "How can I learn to speak Spanish quickly?",
]

results = idx.search(query_text=QUERIES, k=3)
for query, hits in zip(QUERIES, results):
    print(f"\nQuery: '{query}'")
    for r in hits:
        print(f"  rank={r.rank}  score={r.score:.4f}  →  {id_to_text[r.id]}")


Query: 'How do I invest in the stock market?'
  rank=0  score=0.7828  →  What is the step by step guide to invest in share market?
  rank=1  score=0.7230  →  What is the step by step guide to invest in share market in india?
  rank=2  score=0.6286  →  How can I best invest ₹5000 over the next 6 months?

Query: 'What are the best ways to lose weight?'
  rank=0  score=0.6171  →  What is the best diet for a growing decathlete?
  rank=1  score=0.4989  →  How do I prevent breast cancer?
  rank=2  score=0.4861  →  How do I utilize free time to avoid depression?

Query: 'How can I learn to speak Spanish quickly?'
  rank=0  score=0.6843  →  How can I learn to speak English fluently?
  rank=1  score=0.6586  →  How I can speak English fluently?
  rank=2  score=0.5876  →  What are the best ways to learn French?


## Update and delete

| Method | Effect |
|---|---|
| `update(ids, texts=...)` | Re-embed and replace the stored vector for existing IDs (or pass `vectors=` directly) |
| `delete(ids)` | Remove documents from the index |

In [7]:
print(f"Before: {idx.size} documents")

# Rewrite the first document; the embedder re-embeds the new text
new_text = "What is the easiest programming language for a complete beginner?"
idx.update(ids=[all_ids[0]], texts=[new_text])
id_to_text[all_ids[0]] = new_text
print(f"After update: {idx.size} documents  (size unchanged — update is in-place)")

# Delete the next two documents
idx.delete(ids=list(all_ids[1:3]))
print(f"After deleting 2 documents: {idx.size} documents")

Before: 500 documents
After update: 500 documents  (size unchanged — update is in-place)
After deleting 2 documents: 498 documents


## Save and reload

`save(directory)` writes the index to disk. `SimlarEngine.load(directory)` reconstructs it without re-embedding the corpus. Embedders are not saved, so pass `embedder=` when loading.
The reloaded index supports all the same operations — search, update, delete, and further adds.

In [8]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    save_dir = str(Path(tmp) / "quora_index")

    idx.save(save_dir)
    print(f"Saved index to {save_dir}")
    print(f"Files: {[p.name for p in Path(save_dir).iterdir()]}")

    reloaded = SimlarEngine.load(save_dir, embedder=embedder)
    print(f"\nReloaded — size: {reloaded.size}  is_trained: {reloaded.is_trained}")

    # Verify search produces the same top result
    orig_top     = idx.search(query_text=QUERY_TEXT, k=1)[0]
    reloaded_top = reloaded.search(query_text=QUERY_TEXT, k=1)[0]
    match = orig_top.id == reloaded_top.id
    print(f"Top result matches original: {match}")
    print(f"  id={reloaded_top.id}  score={reloaded_top.score:.4f}  →  {id_to_text.get(reloaded_top.id)}")

Saved index to /tmp/tmpn_6m797d/quora_index
Files: ['ids.json', 'config.json', 'index_meta.json', 'bytes', 'coreindex']

Reloaded — size: 498  is_trained: True
Top result matches original: True
  id=1  score=0.8779  →  What is the easiest programming language for a complete beginner?
